# 04 Train MobileNetV2 (ImageNet)
Stage 1: head only, Adam 1e-3, 8 epochs. Stage 2: unfreeze_top(30), Adam 1e-5, up to 15 epochs.


In [ ]:
# 1) Mount Google Drive (a popup asks permission)
from google.colab import drive
drive.mount("/content/drive")

# 2) Paths
DRIVE = "/content/drive/MyDrive/sign-language"

# 3) Code + packages
!git clone https://github.com/<you>/sign-language-translator.git /content/repo 2>/dev/null || git -C /content/repo pull
!pip install -q mediapipe==0.10.14 "numpy<2" scikit-learn seaborn tqdm
%cd /content/repo
import sys
sys.path.insert(0, "/content/repo")



In [ ]:
!mkdir -p /content/custom_alphabet
!cp {DRIVE}/processed/processed_alphabet.zip /content/ && unzip -q /content/processed_alphabet.zip -d /content
!cp {DRIVE}/custom/alphabet.zip /content/ && unzip -q /content/alphabet.zip -d /content/custom_alphabet || echo "no custom alphabet zip yet"



In [ ]:
import json, shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report

from src.config import ALPHABET_LABELS, MOBILENET_IMG_SIZE
from src.models import build_mobilenet, unfreeze_top
from src.train_utils import class_weights_if_imbalanced, default_callbacks, plot_confusion_matrix, plot_history

KAGGLE = Path("/content/processed")
CUSTOM = Path("/content/custom_alphabet")
MIXED = Path("/content/mixed_alphabet")
if MIXED.exists():
    shutil.rmtree(MIXED)
MIXED.mkdir()
rng = np.random.default_rng(0)
own_test_dir = Path("/content/own_test"); shutil.rmtree(own_test_dir, ignore_errors=True); own_test_dir.mkdir()

for lab in ALPHABET_LABELS:
    (MIXED / lab).mkdir(parents=True, exist_ok=True)
    (own_test_dir / lab).mkdir(parents=True, exist_ok=True)
    kfiles = list((KAGGLE / lab).glob("*.jpg"))
    cfiles = list(CUSTOM.rglob(f"{lab}/*.jpg")) + list((CUSTOM / lab).glob("*.jpg"))
    rng.shuffle(cfiles)
    n_hold = max(1, int(0.2 * len(cfiles))) if cfiles else 0
    hold, train_custom = cfiles[:n_hold], cfiles[n_hold:]
    for i, f in enumerate(hold):
        shutil.copy(f, own_test_dir / lab / f"own_{i}.jpg")
    for i, f in enumerate(kfiles + train_custom):
        shutil.copy(f, MIXED / lab / f"{i}.jpg")

SIZE = MOBILENET_IMG_SIZE
ds = tf.keras.utils.image_dataset_from_directory(
    MIXED, labels="inferred", label_mode="int", class_names=ALPHABET_LABELS,
    image_size=(SIZE, SIZE), batch_size=64, shuffle=True, seed=0, color_mode="rgb",
    validation_split=0.15, subset="both",
)
train_raw, val_ds = ds if isinstance(ds, tuple) else (ds, ds)
n = train_raw.cardinality().numpy()
n_test = max(1, int(n * 0.15 / 0.85))
test_ds = train_raw.take(n_test).prefetch(tf.data.AUTOTUNE)
train_ds = train_raw.skip(n_test).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

y_train = np.concatenate([y.numpy() for _, y in train_ds])
cw = class_weights_if_imbalanced(y_train)

model = build_mobilenet(len(ALPHABET_LABELS), SIZE)
model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=cw)

unfreeze_top(model, 30)
model.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
cbs = default_callbacks(f"{DRIVE}/models/mobilenet.keras", patience=6)
history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs, class_weight=cw)

Path(f"{DRIVE}/reports").mkdir(parents=True, exist_ok=True)
plot_history(history, f"{DRIVE}/reports/history_mobilenet.png")
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=ALPHABET_LABELS))
plot_confusion_matrix(y_true, y_pred, ALPHABET_LABELS, f"{DRIVE}/reports/cm_mobilenet.png")
print("params", model.count_params())

own = tf.keras.utils.image_dataset_from_directory(
    own_test_dir, labels="inferred", label_mode="int", class_names=ALPHABET_LABELS,
    image_size=(SIZE, SIZE), batch_size=64, shuffle=False, color_mode="rgb",
)
own_true = np.concatenate([y.numpy() for _, y in own])
own_pred = np.argmax(model.predict(own), axis=1)
print("own webcam test")
print(classification_report(own_true, own_pred, target_names=ALPHABET_LABELS, zero_division=0))

from src.config import PHRASE_LABELS
Path(f"{DRIVE}/models").mkdir(parents=True, exist_ok=True)
with open(f"{DRIVE}/models/labels.json", "w") as f:
    json.dump({"alphabet": ALPHABET_LABELS, "phrases": PHRASE_LABELS}, f)
model.save(f"{DRIVE}/models/mobilenet.keras")

